# 06 — Track-2 production check: records read against their source text

**Purpose.** Human-readable spot-check of the assembled corpus
(`stance_records.parquet`, notebook 08): for sampled records, the evidence
sentences are looked up through the stored indices (record → turn →
sentence list from notebook 01) and printed next to the topic and stance.
Because evidence is selected by index, what is printed is exactly what the
model saw and cited — the check verifies reading quality, not existence.

**Sections.** (i) a seeded uniform sample of 20 — the honest average view;
(ii) ten records per stance pole — does each read as "the member supports
[topic]" / "is concerned about [topic]"; (iii) five lint-flagged records —
what the directional-word review queue contains.

**Verdict (researcher reading, 2026-08-22).** Roughly 85–90% of sampled
records are clean on all three fields. Three defect patterns observed,
each rare and specific:

1. **Stance-target inversion residual** (1 of 20): evidence "eliminating
   conflicts of interests … can go a long way" recorded as topic
   *conflicts of interests* + SUPPORT — the topic should be the reform,
   not the harm. To be quantified in the extraction gold set.
2. **Vague topics** (*agencies*, *cultural problem*): structurally valid,
   low information — the taxonomy's cohesion flag handles these.
3. **Minimal evidence with window-context topics**: a record citing only
   "I am on that bill." with topic *infrastructure bank* — the topic is
   grounded in adjacent (uncited) window sentences. Not hallucination,
   but the cited evidence alone does not verify the topic; the gold
   coding sheet gains an evidence-sufficiency item for this.

**Inputs.** `interim/topics/track2/stance_records.parquet`,
`member_turns.parquet`; hearing titles for context.

**Outputs.** None — a reading artifact. The executed outputs below are the
record of what was checked.

In [1]:
import sys

sys.path.insert(0, "../lib")

import pandas as pd

from paths import DATA_REPO, topics_run_dir

SEED = 7
TRACK2 = topics_run_dir("track2")

records = pd.read_parquet(TRACK2 / "stance_records.parquet")
turns = pd.read_parquet(TRACK2 / "member_turns.parquet",
                        columns=["packageId", "turn_idx", "sentences"])
titles = pd.read_parquet(
    DATA_REPO / "cleaned" / "01_text_summarization" / "hearing_summaries.parquet",
    columns=["packageId", "title"]).set_index("packageId")["title"]
sent = {(p, t): s for p, t, s in
        zip(turns["packageId"], turns["turn_idx"], turns["sentences"])}
print(f"{len(records):,} records, {len(sent):,} turns loaded")

def show(df, header):
    print("=" * 78)
    print(header)
    print("=" * 78)
    for r in df.itertuples():
        who = f"{r.honorific or ''} {r.surname or ''}".strip()
        ev = " ".join(sent[(r.packageId, r.turn_idx)][i - 1]
                      for i in r.sentences)
        print(f"\n--- {r.packageId} ({r.year} {r.chamber}) — {who}"
              f"{' [chair]' if r.is_chair else ''}")
        print(f"    hearing : {str(titles.get(r.packageId, ''))[:90]}")
        print(f"    evidence: {ev[:400]}")
        print(f"    topic   : {r.topic}")
        print(f"    stance  : {r.stance.upper()}"
              + (f"   [lint: {r.topic_lint}]" if r.topic_lint else ""))

32,385 records, 221,311 turns loaded


## A seeded random sample

Twenty records drawn uniformly — the honest view of average quality, not a
curated showcase.

In [2]:
show(records.sample(20, random_state=SEED), "RANDOM SAMPLE (n=20)")

RANDOM SAMPLE (n=20)

--- CHRG-107shrg88291 (2001 SENATE) — Senator Cleland
    hearing : ELECTION REFORM
    evidence: I am an author of legislation to provide a knock-out punch to the punch card system, not by putting a burden on counties, but by putting some federal funds to a third of the precincts in America that have that system, or are now afflicted with that system.
    topic   : federal funds for punch card system reform
    stance  : SUPPORT

--- CHRG-117shrg55821 (2022 SENATE) — Senator Moran
    hearing : NOMINATION TO THE AMTRAK BOARD OF DIRECTORS AND THE SURFACE TRANSPORTATION BOARD
    evidence: And I am disappointed a bit in the response that I received or that several of us received from you in regard to your commitment to long distance passenger service.
    topic   : long distance passenger service
    stance  : CONCERN

--- CHRG-106shrg79713 (2000 SENATE) — Senator McCain
    hearing : S. 2340, AMATEUR SPORTS INTEGRITY ACT AND GAMBLING IN AMATEUR SPORTS
    evidence

## Both stances, deliberately

Ten support and ten concern records, to check each pole reads sensibly as
"the member supports [topic]" / "the member is concerned about [topic]".

In [3]:
show(records[records["stance"] == "support"].sample(10, random_state=SEED),
     "SUPPORT (n=10)")
show(records[records["stance"] == "concern"].sample(10, random_state=SEED),
     "CONCERN (n=10)")

SUPPORT (n=10)

--- CHRG-116shrg54131 (2020 SENATE) — The Chairman [chair]
    hearing : DOES SECTION 230'S SWEEPING IMMUNITY ENABLE BIG TECH BAD BEHAVIOR?
    evidence: This legislation would address the challenges we have discussed while still leaving fundamentals of Section 230 in place.
    topic   : Section 230
    stance  : SUPPORT

--- CHRG-113shrg88868 (2013 SENATE) — Senator Blumenthal
    hearing : CRAFTING A SUCCESSFUL INCENTIVE AUCTION: STAKEHOLDERS' PERSPECTIVES
    evidence: And so I am hoping, and I am going to ask this question of Mr. Epstein, that the FCC will pursue a spectrum policy that best enables competitive forces to benefit consumers and will adopt the Department of Justice recommendations or the kinds of screens or caps that limit the amount of spectrum that any one company can have, simply so that there can be more competition.
    topic   : spectrum policy
    stance  : SUPPORT

--- CHRG-115shrg28382 (2017 SENATE) — Senator Hassan
    hearing : THE PROMISES 

## What the lint flags

Five records carrying a directional-word flag — the review queue. Expected:
mostly proper names ("Environmental Protection Agency") rather than true
rule violations.

In [4]:
show(records[records["topic_lint"].notna()].sample(5, random_state=SEED),
     "LINT-FLAGGED (n=5)")

LINT-FLAGGED (n=5)

--- CHRG-110shrg35959 (2007 SENATE) — Senator Lautenberg
    hearing : HALLIBURTON AND U.S. BUSINESS TIES TO IRAN
    evidence: Mr. Chairman, we've got to stop this behavior. That's why I reintroduced my bill, Stop Business with Terrorists Act last week. The bill will close the loophole that Halliburton has exploited in order to do business with Iran.
    topic   : Stop Business with Terrorists Act
    stance  : SUPPORT   [lint: Stop]

--- CHRG-110shrg77867 (2007 SENATE) — Senator Vitter
    hearing : POLICY IMPLICATIONS OF PHARMACEUTICAL IMPORTATION FOR U.S. CONSUMERS
    evidence: I'm happy to say we put an end to that practice--first, with this amendment on an appropriations bill, and that led to the current USTR abandoning, and stating very clearly to me and others that she would abandon this practice of trying to negotiate those provisions in trade deals.
    topic   : practice of inserting anti-reimportation language through bilateral trade agreements
    stan